In [1]:
# ═══════════════════════════════════════
# TRY 2 — STANDARD SESSION OPENER
# ═══════════════════════════════════════

import json, os, pickle, random, warnings
import numpy as np
import pandas as pd

from sklearn.metrics import precision_score, recall_score, f1_score
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = "/kaggle/input/datasets/aadityaupadhyay1353"

CFG_PATH = f"{BASE}/cps-config/config.json"

with open(CFG_PATH) as f:
    CFG = json.load(f)

TRAIN_IDS = CFG["engine_partition"]["train"]
VAL_IDS = CFG["engine_partition"]["validation"]
TEST_IDS = CFG["engine_partition"]["test"]

OUT = "/kaggle/working/"
os.makedirs(OUT, exist_ok=True)

print("TRY 2 session ready")
print("Train engines:", len(TRAIN_IDS))
print("Val engines:", len(VAL_IDS))
print("Test engines:", len(TEST_IDS))

TRY 2 session ready
Train engines: 70
Val engines: 15
Test engines: 15


In [2]:
df = pd.read_csv(
    f"{BASE}/v2-phase3-features/v2_phase3_feature_dataset.csv"
)

print("Dataset shape:", df.shape)

with open(f"{BASE}/v2-phase3-features/v2_feature_cols.json") as f:
    feature_cols = json.load(f)

print("Total features:", len(feature_cols))

scaler = pickle.load(
    open(f"{BASE}/v2-phase3-features/v2_scaler.pkl", "rb")
)

assert "max_cycle" not in feature_cols

print("Feature set verified — no leakage")

Dataset shape: (20631, 65)
Total features: 56
Feature set verified — no leakage


In [3]:
train_df = df[df["engine_id"].isin(TRAIN_IDS)]
val_df = df[df["engine_id"].isin(VAL_IDS)]

print("Train rows:", train_df.shape)
print("Validation rows:", val_df.shape)

Train rows: (14130, 65)
Validation rows: (3210, 65)


In [4]:
X_train = scaler.transform(train_df[feature_cols])
X_val = scaler.transform(val_df[feature_cols])

print("Feature scaling complete")

Feature scaling complete


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [6]:
results = []

trained_models = {}

for W in CFG["window_sizes"]:

    print(f"\n===== Window: {W} =====")

    y_train = train_df[f"label_w{W}"].values
    y_val = val_df[f"label_w{W}"].values

    neg, pos = np.bincount(y_train)

    scale_pos_weight = neg / pos

    models = {

        "logistic": LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            random_state=SEED
        ),

        "random_forest": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=SEED
        ),

        "xgboost": XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=SEED,
            verbosity=0
        )

    }

    for model_name, model in models.items():

        print("Training:", model_name)

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        precision = precision_score(y_val, preds)
        recall = recall_score(y_val, preds)
        f1 = f1_score(y_val, preds)

        results.append({

            "window": W,
            "model": model_name,
            "precision": precision,
            "recall": recall,
            "f1": f1

        })

        trained_models[f"{model_name}_w{W}"] = model


===== Window: 10 =====
Training: logistic
Training: random_forest
Training: xgboost

===== Window: 20 =====
Training: logistic
Training: random_forest
Training: xgboost

===== Window: 30 =====
Training: logistic
Training: random_forest
Training: xgboost


In [7]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values("f1", ascending=False)

results_df

,window,model,precision,recall,f1
4,20,random_forest,0.873377,0.853968,0.863563
7,30,random_forest,0.869469,0.845161,0.857143
6,30,logistic,0.709321,0.965591,0.817851
1,10,random_forest,0.839744,0.793939,0.816199
5,20,xgboost,0.708235,0.955556,0.813514
8,30,xgboost,0.701954,0.926882,0.798888
2,10,xgboost,0.663866,0.957576,0.784119
3,20,logistic,0.649789,0.977778,0.780735
0,10,logistic,0.578947,1.000000,0.733333


In [8]:
results_df.to_csv(
    f"{OUT}v2_model_baseline_results.csv",
    index=False
)

print("Saved baseline results")

Saved baseline results


In [9]:
top_models = results_df.head(3)

top_models.to_csv(
    f"{OUT}v2_top_models.csv",
    index=False
)

print("Top models:")

print(top_models)

Top models:
   window          model  precision    recall        f1
4      20  random_forest   0.873377  0.853968  0.863563
7      30  random_forest   0.869469  0.845161  0.857143
6      30       logistic   0.709321  0.965591  0.817851


In [10]:
for key, model in trained_models.items():

    pickle.dump(
        model,
        open(f"{OUT}v2_centralized_{key}.pkl", "wb")
    )

print("Saved all trained models")

Saved all trained models


In [11]:
best = top_models.iloc[0]

best_key = f"{best['model']}_w{int(best['window'])}"

pickle.dump(
    trained_models[best_key],
    open(f"{OUT}v2_best_centralized.pkl", "wb")
)

print("Best model:", best_key)

Best model: random_forest_w20


In [12]:
!ls -lh /kaggle/working/

total 32M
---------- 1 root root  21K Apr 12 12:57 __notebook__.ipynb
-rw-r--r-- 1 root root 7.5M Apr 12 12:57 v2_best_centralized.pkl
-rw-r--r-- 1 root root 1.2K Apr 12 12:57 v2_centralized_logistic_w10.pkl
-rw-r--r-- 1 root root 1.2K Apr 12 12:57 v2_centralized_logistic_w20.pkl
-rw-r--r-- 1 root root 1.2K Apr 12 12:57 v2_centralized_logistic_w30.pkl
-rw-r--r-- 1 root root 5.1M Apr 12 12:57 v2_centralized_random_forest_w10.pkl
-rw-r--r-- 1 root root 7.5M Apr 12 12:57 v2_centralized_random_forest_w20.pkl
-rw-r--r-- 1 root root  11M Apr 12 12:57 v2_centralized_random_forest_w30.pkl
-rw-r--r-- 1 root root 292K Apr 12 12:57 v2_centralized_xgboost_w10.pkl
-rw-r--r-- 1 root root 310K Apr 12 12:57 v2_centralized_xgboost_w20.pkl
-rw-r--r-- 1 root root 301K Apr 12 12:57 v2_centralized_xgboost_w30.pkl
-rw-r--r-- 1 root root  649 Apr 12 12:57 v2_model_baseline_results.csv
-rw-r--r-- 1 root root  250 Apr 12 12:57 v2_top_models.csv
